# Cycle Detection and Recovery in BTL and CDM

**Status: notebook design with executed exact-probability checks.**
The simulation, inference, model fitting, and plotting sections below are specifications, not completed experiments.

**Research question:** Can we detect strict pairwise preference cycles from noisy comparisons, distinguish them from sampling artifacts, and quantify how detection improves with data and preference strength?

Build on `notebooks/foundations/Bradley_Terry_Model_MLE/07_BTL_vs_CDM_Interactions_and_Cycles.ipynb` rather than repeating its full derivations. The initial scope is synthetic pairwise observations. Larger-menu context effects and low-rank estimation are extensions.

## 1. What the repository already contains

| Work | Evidence available | What remains for this notebook |
|---|---|---|
| BTL recovery, sampling, MLE existence, optimizer comparisons | Experiment notebooks with saved executions | Use BTL as a negative control for population majority cycles |
| Full CDM recovery and BTL–CDM cross-fits | Experiment notebooks with saved executions | Compare true and estimated preference graphs |
| Unified BTL/full/low-rank comparison | Executed notebook and saved diagnostic tables | Cycle recovery has not been established by those NLL comparisons |
| Paper synthetic recovery and IIA testing | Implementations, executed notebooks, saved numerical results | These answer different questions from cycle detection |
| Three-item cycle foundations | Notebook 07: constructed cycle, BTL containment, pairwise non-identifiability | General detection, uncertainty, and repeated experiments |

These are observations from stored files, not a fresh rerun of earlier experiments. README progress labels lag behind the paper-reproduction artifacts. The graph in `build_cycle_comparison_edges` is an **undirected sampling design**, not a detected preference cycle. Existing directed win-graph diagnostics address MLE existence; they are not a population majority-cycle detector.

## 2. Definitions and mathematical checks

Let $p_{ij}=P(i\mid\{i,j\})$, with $p_{ji}=1-p_{ij}$.

1. **Observed win graph:** add an edge whenever an individual win occurs. Random outcomes under BTL can create cycles here; these do not establish cyclic population preferences.
2. **Strict majority graph:** $i\to j$ iff $p_{ij}>1/2$. A three-cycle has $p_{ij},p_{jk},p_{ki}>1/2$ (or the reverse orientation). Ties produce no edge.
3. **Log-odds inconsistency:** define $\ell_{ij}=\log[p_{ij}/(1-p_{ij})]$ and $c_{ijk}=\ell_{ij}+\ell_{jk}+\ell_{ki}$. BTL has $\ell_{ij}=\theta_i-\theta_j$, so every triangle sum is zero. A nonzero sum can occur even when the majority ordering is transitive.

For a complete comparison graph and interior probabilities, zero log-odds sums on every triangle characterize a BTL score representation. On an incomplete graph, triangles alone do not check all loops.

For CDM, $\ell_{ij}=u_{ij}-u_{ji}$. Cycle detection is based on these identifiable pairwise differences, not on estimating every entry of $U$.

**IIA is separate:** pairwise data alone cannot establish how odds change across menus. Within one fixed menu, CDM softmax utilities give scalar scores whose strict ordering is acyclic. The three pairwise edges of a preference cycle come from three different menus.

In [1]:
from itertools import combinations
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").exists() and (p / "src").is_dir()
)
# Use this checkout explicitly; do not depend on a possibly stale editable install.
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
from bradley_terry.model import sigmoid
from context_dependent_model.model import center_cdm_parameters, cdm_choice_probabilities

SEED = 20261031
ITEMS = ("A", "B", "C")
TOLERANCE = 1e-12

## 3. Exact examples: verify the targets before sampling

For three items, orient the displayed probabilities as $(p_{AB},p_{BC},p_{CA})$.

- BTL: transitive majority ordering and zero triangle sum.
- Symmetric CDM cycle: all three probabilities exceed $1/2$.
- Transitive but non-BTL pairwise probabilities: $(0.8,0.8,0.2)$. No majority cycle, but nonzero log-odds sum.
- Complete indifference: $(0.5,0.5,0.5)$. No strict edges and no strict cycle.

The following cells are the only implemented analysis in this design draft.

In [2]:
theta = np.array([1.0, 0.0, -1.0])
P_btl = sigmoid(theta[:, None] - theta[None, :])

U = torch.tensor([[0., 1., -1.], [-1., 0., 1.], [1., -1., 0.]], dtype=torch.float64)
U = center_cdm_parameters(U)
pairs = list(combinations(range(3), 2))
pair_probs = cdm_choice_probabilities(U, torch.tensor(pairs, dtype=torch.long)).numpy()
P_cdm = np.full((3, 3), 0.5)
for (i, j), probs in zip(pairs, pair_probs):
    P_cdm[i, j], P_cdm[j, i] = probs

P_transitive_non_btl = np.array([[.5, .8, .8], [.2, .5, .8], [.2, .2, .5]])
P_ties = np.full((3, 3), 0.5)
examples = {"BTL": P_btl, "CDM cycle": P_cdm,
            "Transitive, non-BTL": P_transitive_non_btl, "Ties": P_ties}

rows = []
for name, P in examples.items():
    assert np.allclose(P + P.T, 1.0)
    oriented = P[[0, 1, 2], [1, 2, 0]]
    forward = bool(np.all(oriented > 0.5 + TOLERANCE))
    reverse = bool(np.all(oriented < 0.5 - TOLERANCE))
    rows.append({"Example": name, "p_AB": oriented[0], "p_BC": oriented[1],
                 "p_CA": oriented[2], "Strict cycle": forward or reverse,
                 "Log-odds triangle sum": float(np.log(oriented / (1 - oriented)).sum())})
exact_results = pd.DataFrame(rows).set_index("Example")
assert exact_results["Strict cycle"].to_dict() == {
    "BTL": False, "CDM cycle": True, "Transitive, non-BTL": False, "Ties": False}
assert abs(exact_results.loc["BTL", "Log-odds triangle sum"]) < TOLERANCE
assert abs(exact_results.loc["Transitive, non-BTL", "Log-odds triangle sum"]) > 1
assert np.allclose(P_cdm[[0, 1, 2], [1, 2, 0]], sigmoid(2.0))
display(exact_results)

,p_AB,p_BC,p_CA,Strict cycle,Log-odds triangle sum
Example,,,,,
BTL,0.731059,0.731059,0.119203,False,0.000000
CDM cycle,0.880797,0.880797,0.880797,True,6.000000
"Transitive, non-BTL",0.800000,0.800000,0.200000,False,1.386294
Ties,0.500000,0.500000,0.500000,False,0.000000


## 4. Planned reusable detector and graph outputs

Implement reusable functions in a proposed `src/context_dependent_model/cycles.py`; notebooks retain configuration, figures, and interpretation. These functions do not exist yet.

| Proposed function | Input | Output |
|---|---|---|
| `pairwise_probabilities_from_cdm` | Centered interaction matrix | Matrix of pair-menu probabilities |
| `aggregate_pairwise_wins` | Integer `(winner, loser)` observations and item count | Directed win counts and unordered exposure counts |
| `build_preference_graph` | Probability estimates, observed mask, threshold | Directed graph; missing edges remain unknown |
| `find_cyclic_triads` | Directed graph | One canonical row per unordered cyclic triple, with orientation |
| `summarize_preference_cycles` | Directed graph | Triad count, eligible-triad count, DAG status, one cycle witness, strongly connected components |
| `triangle_log_odds` | Interior pair probabilities and observed mask | Signed triangle sums; boundary handling explicit |
| `confidence_preference_graph` | Pairwise win counts and alpha | Simultaneous confidence intervals and supported edges |

Use NetworkX for DAG checks, a cycle witness, and strongly connected components. Count triads by unordered triples to avoid rotation duplicates. Do not enumerate every simple cycle: their number can grow rapidly. Missing comparisons and exact empirical ties create no directed edge; neither is evidence of indifference in the population. Report eligibility/coverage denominators.

For a complete strict tournament, triangle detection suffices to decide whether any directed cycle exists. With missing edges or ties, additionally check general directed cycles: a four-cycle can exist without any observed triangle. Include a four-item directed ring as an algorithm check.

## 5. Planned synthetic generation

Start with known ground-truth probabilities, independently sample pairwise outcomes, and preserve the pair sampling design. Do not turn one larger-menu winner into independent pairwise observations.

**Three-item demonstration:** compare BTL, weak BTL, indifference, transitive non-BTL, and planted CDM cycles. For a planted cycle with edge probability $q=1/2+\delta$, let $a=\operatorname{logit}(q)/2$ and set

$$U=\begin{pmatrix}0&a&-a\\-a&0&a\\a&-a&0\end{pmatrix}.$$

**Larger systems:** embed the same cyclic triple among 6 or 10 items and set remaining interactions to zero for the first controlled experiment. This creates ties outside the planted triple, which must be disclosed. A later background-strength experiment can use $u_{ij}=-\theta_j+\gamma K_{ij}$ for a specified skew matrix $K$. Recompute the actual population graph rather than assuming a chosen $\gamma$ creates a cycle.

For each unordered pair, generate an independent Bernoulli stream. Use nested prefixes for growing sample sizes within a run; use independent seeds across runs. Report total comparisons as $m\binom{n}{2}$, where $m$ is comparisons **per pair**.

In [3]:
# Proposed settings only: this cell does not launch experiments.
PILOT_DESIGN = {
    "seed": SEED,
    "number_of_items": [3],
    "comparisons_per_pair": [10, 50, 200, 1000],
    "cycle_probability_margins": [0.05, 0.15, 0.30],
    "independent_runs": 20,
    "familywise_alpha": 0.05,
}
EXTENSION_DESIGN = {
    "number_of_items": [6, 10],
    "confirmatory_runs": 200,
    "sampling": ["balanced complete", "imbalanced", "incomplete"],
}
PILOT_DESIGN

{'seed': 20261031,
 'number_of_items': [3],
 'comparisons_per_pair': [10, 50, 200, 1000],
 'cycle_probability_margins': [0.05, 0.15, 0.3],
 'independent_runs': 20,
 'familywise_alpha': 0.05}

## 6. Planned inference: empirical cycles versus supported cycles

Report both graphs:

- **Empirical graph:** orient observed pairs by $\hat p_{ij}>1/2$. This is descriptive and can show false cycles due to noise.
- **Confidence graph:** for the predeclared family of $K=\binom{n}{2}$ unordered pairs, use two-sided exact binomial (Clopper–Pearson) intervals at confidence $1-\alpha/K$. Orient $i\to j$ only if the interval for $p_{ij}$ lies entirely above $1/2$; orient the reverse direction if it lies entirely below $1/2$. Otherwise abstain. With zero observations use $[0,1]$.

Bonferroni gives simultaneous edge coverage at least $1-\alpha$ for each predeclared dataset/analysis. Under an acyclic population graph, any directed cycle in this supported graph requires at least one incorrect edge orientation. This supplies a conservative false-cycle control under independent Bernoulli trials within each pair. Validate empirically with BTL and tied controls.

This guarantee is per sample-size analysis; repeatedly inspecting many sample sizes does not provide a sequential guarantee. If making an “ever detected” claim, adjust for all planned looks or design a confidence-sequence procedure separately. The pilot reports each sample size separately.

“No supported cycle” means insufficient evidence or no detected cycle, not proof of acyclicity. Three separate 95% intervals do not automatically provide a 95% simultaneous cycle claim. Do not select a favorable triple and ignore the selection across other pairs.

Log-odds diagnostics are initially descriptive. Empirical probabilities of zero or one produce infinite logits: preserve raw estimates, and label any Jeffreys-smoothed display $(w_{ij}+1/2)/(N_{ij}+1)$ separately. Do not silently clip or treat smoothed curls as significance tests.

## 7. Planned repeated experiments and figures

| Experiment | Purpose | Primary outputs |
|---|---|---|
| Exact examples and detector checks | Verify definitions and graph algorithms | Probabilities, cycle witness, triads, triangle sums |
| BTL and tied controls | Measure spurious empirical cycles and supported false positives | Probability of any detected cycle, with Monte Carlo intervals |
| Planted cycles at varying strength | Measure detection power and edge recovery | Supported-cycle detection versus comparisons per pair |
| Six- and ten-item embeddings | Check localization and multiple-pair adjustment | Cyclic-triple recovery, false cyclic triples, edge abstention |
| Imbalanced/incomplete sampling | Separate coverage limitations from algorithm failure | Eligible triples, coverage, general-cycle witness |

Recommended figures: probability heatmaps; labeled majority graphs; detection rate versus comparisons per pair, separated by signal strength; empirical versus confidence-supported false-cycle rates; and triad count versus log-odds inconsistency for the exact controls.

Use Wilson intervals for Monte Carlo proportions and report run counts. Twenty runs are a pipeline pilot, not precise calibration. Before extending to 200 runs, check pilot correctness, runtime, and null behavior. Summarize each sample size separately because nested prefixes are dependent within a run. Report precision as undefined when no cycles are predicted rather than converting it to a perfect score.

## 8. Optional second phase: BTL versus CDM fitting

After direct detection works, fit BTL and pairwise CDM on the same training comparisons and compare predicted pairwise probabilities, recovered cycle structure, and held-out NLL. A fitted BTL population graph is acyclic by construction; this does not imply the empirical training graph is acyclic.

Pair-only observations identify $d_{ij}=u_{ij}-u_{ji}$, not the full interaction matrix. A pairwise CDM may use the canonical representative $u_{ij}=d_{ij}/2$, $u_{ji}=-d_{ij}/2$, but this is a convention and does not recover the true symmetric interactions. Parameter recovery of all of $U$ is inappropriate here.

Track separation, boundary probabilities, parameter norms, and stopping diagnostics. Do not equate a numerical convergence flag with a finite unregularized MLE. If regularization or smoothing is used, state that it changes the estimator.

Defer low-rank cycle recovery, formal bootstrap tests of BTL goodness of fit, and full Hodge decomposition until the core notebook is stable. A weighted log-odds projection can later study residual inconsistency, but it is not the same objective as BTL maximum likelihood.

## 9. Optional context-effect example

Reuse the construction from Notebook 07 in which two centered CDM matrices have identical pairwise differences but different probabilities on $\{A,B,C\}$. This illustrates why a cycle detector cannot be a complete detector of menu effects.

For an explicit IIA comparison, compare $\log[P(i\mid C)/P(j\mid C)]$ between two menus containing both items. Keep these menu-conditioned observations separate. Pooling heterogeneous menus can introduce misleading pairwise summaries.

## 10. Outputs and completion criteria

When simulations are implemented, save results under `results/experiments/cycle_detection/<run_id>/`, with matching figure/table directories under `docs/thesis/`. The draft currently writes no research outputs.

Save configuration and package versions; generating parameters and true probabilities; pairwise win/exposure counts; empirical estimates and simultaneous intervals; true, empirical, and supported graphs; cyclic-triple tables; DAG status and cycle witnesses; per-run metrics and aggregate summaries. Any model fits need their own diagnostics.

**Completion criteria:**

1. Exact BTL has zero log-odds triangle sums and no strict majority cycles.
2. The planted CDM cycle is detected once, with the correct orientation; relabeling preserves results.
3. The transitive non-BTL example has no strict cycle despite nonzero triangle sum.
4. Missing pairs, ties, all-win/all-loss pairs, and the triangle-free four-cycle behave as specified.
5. Repeated null and planted-cycle experiments report uncertainty and coverage without hiding abstentions or failed fits.
6. The completed notebook runs from a fresh kernel and all reported conclusions refer to actual outputs.

**Next implementation step:** write and verify the reusable graph/count routines, then add the three-item sampling and confidence-interval demonstration. No large fitting sweep is needed for that first step.

## References

- Existing project foundation: `07_BTL_vs_CDM_Interactions_and_Cycles.ipynb`.
- Seshadri, Peysakhovich & Ugander (2019), [Discovering Context Effects from Raw Choice Data](https://proceedings.mlr.press/v97/seshadri19a.html): CDM and menu effects.
- Jiang, Lim, Yao & Ye, [Statistical ranking and combinatorial Hodge theory](https://arxiv.org/abs/0811.1067): global ranking components and residual cyclic inconsistency. This motivates a possible later extension; full Hodge decomposition is not implemented in this draft.

The statistical experiment design above is proposed for this project; it is not claimed to reproduce a cycle-detection experiment from either paper.